In [2]:
import re
import json
import subprocess
import requests
import os
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from collections import Counter

In [3]:
class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    # API helpers (from HW2)
    def _http_function(self, endpoint, **kwargs):
        """Generic HTTP GET function returning response object."""
        headers = kwargs.get('headers', {})
        if 'User-Agent' not in headers:
            headers['User-Agent'] = 'HW4/1.0 (student@example.com)'
        kwargs['headers'] = headers
        response = requests.get(endpoint, **kwargs)
        return response

    def _get_uniprot(self, accession):
        """Get UniProt entry for a single accession."""
        endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"
        return self._http_function(endpoint, headers={'Accept': 'application/json'})

    def _get_ensembl(self, id):
        """Get Ensembl entry for a single ID."""
        endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
        headers = {'Content-Type': 'application/json'}
        return self._http_function(endpoint, headers=headers)

    def _uniprot_parse_response(self, resp):
        """Parse UniProt JSON response."""
        try:
            data = resp.json()
        except Exception as e:
            return {'error': f'Failed to parse JSON: {str(e)}'}

        try:
            organism = data.get('organism', {}).get('scientificName', 'N/A')
            geneInfo = data.get('genes', [])
            seq = data.get('sequence', {})
            sequenceInfo = {
                'value': seq.get('value', ''),
                'length': seq.get('length', 'N/A'),
                'molWeight': seq.get('molWeight', 'N/A'),
                'crc64': seq.get('crc64', 'N/A'),
                'md5': seq.get('md5', 'N/A')
            }
            entry_type = 'protein'
            return {
                'organism': organism,
                'geneInfo': geneInfo,
                'sequenceInfo': sequenceInfo,
                'type': entry_type
            }
        except Exception as e:
            return {'error': f'UniProt parsing failed: {str(e)}'}

    def _ensembl_parse_response(self, resp):
        """Parse Ensembl JSON response."""
        try:
            data = resp.json()
        except Exception as e:
            return {'error': f'Failed to parse JSON: {str(e)}'}

        try:
            if 'error' in data:
                return {'error': data['error']}
            return {
                'object_type': data.get('object_type', 'N/A'),
                'assembly_name': data.get('assembly_name', 'N/A'),
                'species': data.get('species', 'N/A'),
                'db_type': data.get('db_type', 'N/A'),
                'biotype': data.get('biotype', 'N/A'),
                'display_name': data.get('display_name', 'N/A'),
                'id': data.get('id', 'N/A'),
                'description': data.get('description', 'N/A'),
                'canonical_transcript': data.get('canonical_transcript', 'N/A'),
                'source': data.get('source', 'N/A')
            }
        except Exception as e:
            return {'error': f'Ensembl parsing failed: {str(e)}'}

    def _access_database(self, id, database, seq_description, seq_sequence):
        """
        Call the appropriate API and parse the response.
        Returns a dictionary with database info.
        """
        if database == 'uniprot':
            resp = self._get_uniprot(id)
            info = self._uniprot_parse_response(resp)
            return {'DB_name': 'uniprot', 'info': info}
        elif database == 'ensembl':
            resp = self._get_ensembl(id)
            info = self._ensembl_parse_response(resp)
            return {'DB_name': 'ensembl', 'info': info}
        else:
            return {'error': f'Unknown database: {database}'}

    # fallback stats with biopython
    def _compute_stats_biopython(self):
        """Compute FASTA statistics using Biopython (fallback if seqkit not found)."""
        sequences = list(SeqIO.parse(self.filename, 'fasta'))
        if not sequences:
            return {'error': 'No sequences found in file'}

        lengths = [len(rec.seq) for rec in sequences]
        lengths_sorted = sorted(lengths)
        n = len(lengths_sorted)

        # Basic stats
        total_len = sum(lengths)
        min_len = min(lengths)
        max_len = max(lengths)
        avg_len = total_len / n

        # Quartiles
        q1_index = n // 4
        q2_index = n // 2
        q3_index = 3 * n // 4
        q1 = lengths_sorted[q1_index]
        q2 = lengths_sorted[q2_index] if n % 2 == 0 else lengths_sorted[q2_index]
        q3 = lengths_sorted[q3_index]

        # N50
        half_total = total_len / 2
        cum = 0
        n50 = 0
        for l in lengths_sorted[::-1]:
            cum += l
            if cum >= half_total:
                n50 = l
                break
        n50_num = sum(1 for l in lengths if l >= n50)

        # GC content
        all_seq = ''.join(str(rec.seq) for rec in sequences)
        gc_count = all_seq.count('G') + all_seq.count('C')
        gc_percent = (gc_count / len(all_seq) * 100) if all_seq else 0

        # type (DNA/Protein)
        allowed_dna = set('ATGCUatgcuNn')
        is_dna = True
        for rec in sequences:
            if any(ch not in allowed_dna for ch in str(rec.seq).upper()):
                is_dna = False
                break
        seq_type = 'DNA' if is_dna else 'Protein'

        return {
            'fasta_seqkit_stat_info': {
                'format': 'FASTA',
                'type': seq_type,
                'num_seqs': n,
                'sum_len': total_len,
                'min_len': min_len,
                'avg_len': round(avg_len, 2),
                'max_len': max_len,
                'Q1': q1,
                'Q2': q2,
                'Q3': q3,
                'sum_gap': 0,   # not computed
                'N50': n50,
                'N50_num': n50_num,
                'Q20(%)': 0,    # not relevant for FASTA
                'Q30(%)': 0,
                'AvgQual': 0,
                'GC(%)': round(gc_percent, 2),
                'sum_n': all_seq.upper().count('N')
            },
            'fasta_type': seq_type,
            'fasta_num_seqs': n
        }

    def seqkit_stats(self):
        """Call seqkit stats; if seqkit not found, use biopython fallback."""
        try:
            result = subprocess.run(
                ['seqkit', 'stats', self.filename],
                capture_output=True,
                text=True,
                check=False
            )
            if result.returncode != 0:
                # seqkit returned an error – fall back to biopython
                return self._compute_stats_biopython()

            # parse the tab-separated output
            lines = result.stdout.strip().split('\n')
            if len(lines) < 2:
                return self._compute_stats_biopython()
            
            # the header line: file format type num_seqs sum_len min_len avg_len max_len
            header = lines[0].split('\t')
            data = lines[1].split('\t')
            stats = dict(zip(header, data))

            # determine file type: "DNA" or "Protein" from the 'type' column
            file_type = stats.get('type', 'Unknown').strip()
            if file_type not in ('DNA', 'Protein', 'RNA'):
                file_type = 'Unknown'

            numeric_fields = ['num_seqs', 'sum_len', 'min_len', 'avg_len', 'max_len',
                              'Q1', 'Q2', 'Q3', 'sum_gap', 'N50', 'N50_num',
                              'Q20(%)', 'Q30(%)', 'AvgQual', 'GC(%)', 'sum_n']
            for field in numeric_fields:
                if field in stats:
                    try:
                        stats[field] = int(stats[field]) if '.' not in stats[field] else float(stats[field])
                    except:
                        pass

            return {
                'fasta_seqkit_stat_info': stats,
                'fasta_type': file_type,
                'fasta_num_seqs': int(stats.get('num_seqs', 0))
            }

        except FileNotFoundError:
            # seqkit not installed – use Biopython fallback
            return self._compute_stats_biopython()
        except Exception as e:
            return {'error': str(e)}

    def biopython_parser(self, seqkit_result):
        """
        Parse FASTA file with Biopython, extract IDs from descriptions,
        and call the appropriate database.
        Returns dictionary with results.
        """
        # determine which regex to use based on file type
        file_type = seqkit_result.get('fasta_type', 'Unknown')
        if file_type == 'Protein':
            # uniProt pattern: [A-Z][0-9][A-Z0-9]{3,}[0-9]
            id_pattern = re.compile(r'[A-Z][0-9][A-Z0-9]{3,}[0-9]')
            database = 'uniprot'
        elif file_type == 'DNA':
            # ensembl pattern: ENS[A-Z]*[GTEP][0-9]{11}
            id_pattern = re.compile(r'ENS[A-Z]*[GTEP][0-9]{11}')
            database = 'ensembl'
        else:
            id_pattern = None
            database = None

        output = {}
        warnings = []

        try:
            for record in SeqIO.parse(self.filename, 'fasta'):
                desc = record.description
                seq = str(record.seq)

                # for known ID patterns
                id_found = None
                if id_pattern:
                    # pre-determined pattern
                    match = id_pattern.search(desc)
                    if match:
                        id_found = match.group()
                else:
                    # Try both patterns if type unknown
                    uniprot_pat = re.compile(r'[A-Z][0-9][A-Z0-9]{3,}[0-9]')
                    ensembl_pat = re.compile(r'ENS[A-Z]*[GTEP][0-9]{11}')
                    match = uniprot_pat.search(desc)
                    if match:
                        id_found = match.group()
                        database = 'uniprot'
                    else:
                        match = ensembl_pat.search(desc)
                        if match:
                            id_found = match.group()
                            database = 'ensembl'

                if id_found and database:
                    # call the API
                    result = self._access_database(id_found, database, desc, seq)
                    if 'error' in result.get('info', {}):
                        warnings.append({'No ID match found.'})
                    else:
                        output[f'file_info_{id_found}'] = {
                            'description': desc,
                            'sequence': seq
                        }
                        output[f'database_info_{id_found}'] = result['info']
                else:
                    # no ID found
                    warnings.append({'No ID match found.'})

            if database:
                output['DB_name'] = database

            # Add warning(s) as a single dict if exactly one, else as list
            if warnings:
                if len(warnings) == 1:
                    output['WARNING'] = warnings[0]
                else:
                    output['WARNING'] = warnings

            return output

        except Exception as e:
            return {'error': str(e)}

    def show_output(self, output, indent=0):
        """Print the dictionary."""
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [4]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
stats

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': 2,
  'sum_len': 456,
  'min_len': 29,
  'avg_len': 228.0,
  'max_len': 427,
  'Q1': 29,
  'Q2': 427,
  'Q3': 427,
  'sum_gap': 0,
  'N50': 427,
  'N50_num': 1,
  'Q20(%)': 0,
  'Q30(%)': 0,
  'AvgQual': 0,
  'GC(%)': 8.33,
  'sum_n': 14},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 2}

In [5]:
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEED